# AAI-510 Final Project — Notebook 2: Agent Definition
**Owner:** Wael Alhathal (AI Engineer)
**Team:** Group 7 — Wael Alhathal & Idrees Khan
**Project:** Nova — Order & Returns Support Assistant for Nestwell

---

## What this notebook does

This notebook defines Nova, our ReAct-pattern customer support agent. It covers:

1. System prompt and tool definitions
2. The agent loop (ReAct: Reason → Act → Observe → Respond)
3. MLflow tracing setup
4. Running the agent against both LLMs (`databricks-gpt-oss-20b` and `databricks-gpt-oss-120b`)
5. Five complete traces including the dual-LLM comparison trace
6. Two out-of-scope rejection demonstrations

The evaluation notebook (Notebook 3) picks up from here and scores all traces.

## 0. Setup & Imports

In [0]:
%pip install -U -qqqq "mlflow>=3.9" openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from datetime import datetime

import mlflow
import mlflow.deployments

# On MLflow 3.9 this import works straight from mlflow.entities.
# The fallback only triggers on older versions, keeping the notebook portable.
try:
    from mlflow.entities import SpanType
except Exception:
    class SpanType:
        AGENT = "AGENT"
        LLM = "LLM"
        TOOL = "TOOL"
        CHAIN = "CHAIN"
        UNKNOWN = "UNKNOWN"

# Unity Catalog settings (must match Notebook 1)
CATALOG = "main"
SCHEMA  = "nestwell"

# We intentionally do NOT call mlflow.set_experiment("/Users/...").
# On serverless that trips the "modelRegistryUri is not available" error.
# Traces and runs log to this notebook's own MLflow experiment automatically.

print("MLflow version:", mlflow.__version__)
print("Setup complete")

MLflow version: 3.13.0
Setup complete


## 1. Tool Definitions

We define two Python wrappers around our Unity Catalog functions.
The agent calls these; we keep them as thin wrappers so the heavy logic
stays in the registered UC functions (easier to update without changing agent code).

In [0]:
def order_lookup(order_id: str) -> dict:
    """
    Looks up a Nestwell order by order ID.
    Calls the Unity Catalog SQL function registered in Notebook 1.
    Returns a dict the agent can reason over, or an error message if not found.
    """
    try:
        result = spark.sql(f"""
            SELECT * FROM {CATALOG}.{SCHEMA}.order_lookup('{order_id.strip()}')
        """).collect()

        if not result:
            return {
                "found": False,
                "message": f"No order found with ID {order_id}. Please double-check the order ID."
            }

        row = result[0].asDict()
        row["found"] = True
        return row

    except Exception as e:
        return {"found": False, "message": f"Error looking up order: {str(e)}"}


def policy_lookup(topic: str) -> str:
    """
    Returns the Nestwell policy section most relevant to the given topic.
    Calls the Unity Catalog Python function registered in Notebook 1.
    """
    try:
        result = spark.sql(f"""
            SELECT {CATALOG}.{SCHEMA}.policy_lookup('{topic.strip().lower()}') AS section
        """).collect()
        return result[0]["section"]
    except Exception as e:
        return f"Error retrieving policy: {str(e)}"


# Quick smoke test
print("order_lookup test:")
print(order_lookup("ORD-1000"))
print("\npolicy_lookup test:")
print(policy_lookup("return")[:200])

order_lookup test:
{'order_id': 'ORD-1000', 'customer_name': 'Sarah Garcia', 'order_status': 'delivered', 'order_purchase_timestamp': '2023-06-22 12:13:55', 'order_estimated_delivery_date': '2023-06-29', 'order_delivered_customer_date': '2023-06-29 16:13:55', 'return_eligible': False, 'days_since_purchase': 192, 'item_count': 1, 'order_total': 77.84, 'found': True}

policy_lookup test:
SECTION 1 — RETURN ELIGIBILITY
    
    Most items sold by Nestwell can be returned within 30 days of the delivery date,
    provided the item is unused, in its original packaging, and accompanied by 


## 2. Tool Schema for the LLM

We pass the tool definitions in OpenAI function-calling format.
Both Databricks Foundation Model endpoints support this schema.

In [0]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "order_lookup",
            "description": (
                "Look up a Nestwell order by its order ID (format: ORD-XXXX). "
                "Use this when the customer asks about order status, delivery date, "
                "tracking, or return eligibility for a specific order."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID, e.g. ORD-1042"
                    }
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "policy_lookup",
            "description": (
                "Look up Nestwell's return and shipping policy. "
                "Use this when the customer asks about return windows, how to start a return, "
                "refund timelines, shipping costs, damaged items, or exchanges. "
                "Pass a short topic keyword like 'return', 'refund', 'shipping', or 'damage'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "Short keyword describing the policy topic, e.g. 'return', 'refund', 'shipping'"
                    }
                },
                "required": ["topic"]
            }
        }
    }
]

TOOL_MAP = {
    "order_lookup":  order_lookup,
    "policy_lookup": policy_lookup,
}

print("Tool schema ready.")

Tool schema ready.


## 3. System Prompt

The system prompt defines Nova's role, what it can and can't answer,
and how it should handle out-of-scope requests.
We spent a fair amount of time on the out-of-scope wording —
the agent needs to decline clearly without being rude.

In [0]:
SYSTEM_PROMPT = """You are Nova, a friendly and helpful customer support assistant for Nestwell, an online home and lifestyle retailer.

Your job is to answer customer questions about their orders and Nestwell's return and shipping policy. You have access to two tools:

1. order_lookup — use this to look up a specific order by its order ID (format: ORD-XXXX)
2. policy_lookup — use this to look up Nestwell's return, refund, and shipping policies

Guidelines:
- Always use a tool before answering any question about an order or policy. Never guess or make up order details.
- If a customer provides an order ID, call order_lookup first. If the order is not found, say so clearly and ask them to double-check.
- If a customer asks about returns, refunds, shipping costs, or damaged items, call policy_lookup with the relevant topic keyword.
- Some questions require both tools — for example, "can I return order ORD-1042?" needs order_lookup (to check return_eligible) and may also need policy_lookup (to explain the process).
- Be concise and warm. Customers are often frustrated when they contact support, so keep your tone friendly and solution-focused.
- If a question is outside your scope (not about orders or Nestwell policy), politely decline and redirect. Do not attempt to answer questions about unrelated topics.
- Never invent order details, dates, prices, or policy rules. Only use what the tools return.

Out-of-scope examples (decline these politely):
- Questions about other companies or products not sold by Nestwell
- General knowledge questions, coding, writing, math, or anything not related to Nestwell orders/policy
- Requests to write content, poems, or do tasks unrelated to customer support
"""

print("System prompt ready.")

System prompt ready.


## 4. The Agent Loop (ReAct Pattern)

Our agent follows the ReAct pattern:
- **Reason**: the LLM reads the customer message and decides what to do
- **Act**: it calls a tool (order_lookup or policy_lookup)
- **Observe**: we run the tool and feed the result back
- **Respond**: the LLM generates a final grounded answer

We wrap the whole run in an MLflow trace so every step is logged.

In [0]:
def extract_text(content):
    """gpt-oss returns content as a list of blocks (reasoning + text) rather
    than a plain string. This pulls out the human-facing answer text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = [b.get("text", "") for b in content
                 if isinstance(b, dict) and b.get("type") == "text"]
        return "\n".join(p for p in parts if p).strip()
    return str(content)


def run_nova(user_message: str, model_name: str, trace_name: str = "nova_trace") -> str:
    """Runs the Nova agent on one customer message and logs a full MLflow trace."""
    client = mlflow.deployments.get_deploy_client("databricks")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]

    with mlflow.start_span(name=trace_name, span_type=SpanType.AGENT) as root_span:
        root_span.set_inputs({"user_message": user_message, "model": model_name})

        final_response = None
        max_turns = 5

        for turn in range(max_turns):
            with mlflow.start_span(name=f"llm_call_turn_{turn}", span_type=SpanType.LLM) as llm_span:
                llm_span.set_inputs({"messages": messages, "tools": TOOLS})
                response = client.predict(
                    endpoint=model_name,
                    inputs={
                        "messages": messages,
                        "tools": TOOLS,
                        "tool_choice": "auto",
                        "max_tokens": 512,
                        "temperature": 0.0,
                    },
                )
                assistant_msg = response["choices"][0]["message"]
                llm_span.set_outputs({"response": assistant_msg})

            tool_calls = assistant_msg.get("tool_calls") or []

            if not tool_calls:
                final_response = extract_text(assistant_msg.get("content"))
                break

            messages.append({"role": "assistant", **assistant_msg})

            for tc in tool_calls:
                fn_name = tc["function"]["name"]
                fn_args = json.loads(tc["function"]["arguments"])

                with mlflow.start_span(name=f"tool_{fn_name}", span_type=SpanType.TOOL) as tool_span:
                    tool_span.set_inputs({"tool": fn_name, "arguments": fn_args})
                    if fn_name in TOOL_MAP:
                        tool_result = TOOL_MAP[fn_name](**fn_args)
                    else:
                        tool_result = {"error": f"Unknown tool: {fn_name}"}
                    tool_span.set_outputs({"result": tool_result})

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": json.dumps(tool_result),
                })

        if final_response is None:
            final_response = "I'm sorry, I wasn't able to complete that request. Please try again or contact support."

        root_span.set_outputs({"final_response": final_response})

    return final_response

print("Agent loop defined (with gpt-oss content fix).")

Agent loop defined (with gpt-oss content fix).


## 5. Five Traces

We run 5 traces covering:
- Trace 1: Order status lookup (delivered)
- Trace 2: Return eligibility check
- Trace 3: Policy question (shipping cost)
- Trace 4: Out-of-scope rejection #1
- Trace 5: Dual-LLM comparison (same input, both models)
- (Bonus) Out-of-scope rejection #2

In [0]:
# Pick some real order IDs from our data for realistic traces
delivered_orders = spark.sql(f"""
    SELECT order_id, return_eligible FROM {CATALOG}.{SCHEMA}.orders
    WHERE order_status = 'delivered'
    LIMIT 10
""").collect()

shipped_orders = spark.sql(f"""
    SELECT order_id FROM {CATALOG}.{SCHEMA}.orders
    WHERE order_status = 'shipped'
    LIMIT 5
""").collect()

eligible_order   = next(r["order_id"] for r in delivered_orders if r["return_eligible"])
ineligible_order = next(r["order_id"] for r in delivered_orders if not r["return_eligible"])
shipped_order    = shipped_orders[0]["order_id"]

print(f"Eligible for return   : {eligible_order}")
print(f"NOT eligible (old)    : {ineligible_order}")
print(f"Shipped (in transit)  : {shipped_order}")

Eligible for return   : ORD-1007
NOT eligible (old)    : ORD-1000
Shipped (in transit)  : ORD-1001


In [0]:
SMALL_MODEL = "databricks-gpt-oss-20b"
LARGE_MODEL = "databricks-gpt-oss-120b"

### Trace 1 — Order Status (Shipped, In-Transit)

In [0]:
with mlflow.start_run(run_name="trace_1_order_status"):
    q1 = f"Hi, can you tell me the status of my order {shipped_order}? I'm wondering when it will arrive."
    print(f"Question: {q1}\n")
    r1 = run_nova(q1, SMALL_MODEL, trace_name="trace_1_order_status")
    print(f"Nova: {r1}")
    mlflow.log_param("question", q1)
    mlflow.log_param("model", SMALL_MODEL)
    mlflow.log_text(r1, "response.txt")

Question: Hi, can you tell me the status of my order ORD-1001? I'm wondering when it will arrive.

Nova: Hi Michael! Your order **ORD‑1001** is on its way. It was shipped on **November 11, 2023** and is expected to arrive by **November 23, 2023**. If you need any more help or have questions about the items, just let me know!


Trace(trace_id=tr-dd2259028c2b27a07028ba89233d6749)

### Trace 2 — Return Eligibility Check (order that IS eligible)

In [0]:
with mlflow.start_run(run_name="trace_2_return_eligible"):
    q2 = f"I'd like to return order {eligible_order}. Is that possible?"
    print(f"Question: {q2}\n")
    r2 = run_nova(q2, SMALL_MODEL, trace_name="trace_2_return_eligible")
    print(f"Nova: {r2}")
    mlflow.log_param("question", q2)
    mlflow.log_param("model", SMALL_MODEL)
    mlflow.log_text(r2, "response.txt")

Question: I'd like to return order ORD-1007. Is that possible?

Nova: ✅ Great news—your order **ORD‑1007** is return‑eligible.

Here’s what to do next:

1. **Pack the item** in its original packaging (or a suitable box) and keep the receipt or order confirmation handy.
2. **Print a return label**: Log in to your Nestwell account, go to *Orders*, select ORD‑1007, and click “Return Item.”  
3. **Drop it off** at the nearest UPS or USPS location (the label will show the pickup/drop‑off address).

You’ll receive a refund to the original payment method once we receive the package and confirm it’s unused. If you have any questions while packing or shipping, just let me know!


Trace(trace_id=tr-663a565bc704884ee50bafbe2983b7ad)

### Trace 3 — Policy Question (Shipping Costs)

In [0]:
with mlflow.start_run(run_name="trace_3_policy_shipping"):
    q3 = "How much does shipping cost? Is there free shipping?"
    print(f"Question: {q3}\n")
    r3 = run_nova(q3, SMALL_MODEL, trace_name="trace_3_policy_shipping")
    print(f"Nova: {r3}")
    mlflow.log_param("question", q3)
    mlflow.log_param("model", SMALL_MODEL)
    mlflow.log_text(r3, "response.txt")

Question: How much does shipping cost? Is there free shipping?

Nova: Shipping costs vary by the option you choose:

| Shipping option | Cost | Notes |
|------------------|------|-------|
| **Standard Shipping** (5–8 business days) | **Free** on orders over **$75** | $5.99 on orders under $75 |
| **Expedited Shipping** (2–3 business days) | Flat **$12.99** on all orders |
| **Overnight Shipping** (next business day, order by 12 pm PT) | Flat **$24.99** on all orders |

- Shipping times are estimates and don’t include the typical 1‑day order‑processing time.
- Nestwell ships only within the contiguous United States (P.O. Boxes are accepted for Standard Shipping only).
- Shipping fees are non‑refundable unless the return is due to a Nestwell error (e.g., wrong item shipped or damaged on arrival).

Let me know if you’d like to choose a specific shipping method or have any other questions!


Trace(trace_id=tr-ef021eef46114f36fc36ba4e480e3d6c)

### Trace 4 — Out-of-Scope Rejection #1

In [0]:
with mlflow.start_run(run_name="trace_4_out_of_scope_1"):
    q4 = "Can you write me a poem about autumn leaves?"
    print(f"Question: {q4}\n")
    r4 = run_nova(q4, SMALL_MODEL, trace_name="trace_4_oos_poem")
    print(f"Nova: {r4}")
    mlflow.log_param("question", q4)
    mlflow.log_param("model", SMALL_MODEL)
    mlflow.log_text(r4, "response.txt")

Question: Can you write me a poem about autumn leaves?

Nova: I’m sorry, but I can’t help with that. If you have any questions about your Nestwell order or our return and shipping policies, feel free to let me know!


Trace(trace_id=tr-624e561586962d997a30a121000711bf)

### Trace 5 — Dual-LLM Comparison (same question, both models)

This trace runs the same input through both models so we can compare
quality and cost side by side. We use a slightly tricky question —
return eligibility on an order that is NOT eligible — because that's
where we expect the larger model to reason more carefully.

In [0]:
with mlflow.start_run(run_name="trace_5_dual_llm_comparison"):
    q5 = f"I want to return order {ineligible_order}. It's been a while but I really don't like the product."

    print(f"Question: {q5}\n")
    print("=" * 60)
    print(f"Running with {SMALL_MODEL}...")
    r5_small = run_nova(q5, SMALL_MODEL, trace_name=f"trace_5_{SMALL_MODEL}")
    print(f"Nova ({SMALL_MODEL}):\n{r5_small}\n")

    print("=" * 60)
    print(f"Running with {LARGE_MODEL}...")
    r5_large = run_nova(q5, LARGE_MODEL, trace_name=f"trace_5_{LARGE_MODEL}")
    print(f"Nova ({LARGE_MODEL}):\n{r5_large}\n")

    mlflow.log_param("question", q5)
    mlflow.log_param("small_model", SMALL_MODEL)
    mlflow.log_param("large_model", LARGE_MODEL)
    mlflow.log_text(r5_small, "response_small_model.txt")
    mlflow.log_text(r5_large, "response_large_model.txt")

Question: I want to return order ORD-1000. It's been a while but I really don't like the product.

Running with databricks-gpt-oss-20b...
Nova (databricks-gpt-oss-20b):
I’m sorry to hear the product isn’t meeting your expectations. Unfortunately, this order is **not return‑eligible**. Nestwell’s return window is 30 days from the delivery date, and this order was delivered on June 29, 2023—well beyond that period.

If you’d like to discuss other options (such as a store credit or a different product), please let me know and I’ll connect you with the right team. Thank you for shopping with Nestwell!

Running with databricks-gpt-oss-120b...
Nova (databricks-gpt-oss-120b):
Hi Sarah,

I’ve checked your order **ORD‑1000** and, unfortunately, it’s **not return‑eligible**. Nestwell’s return policy allows returns only within 30 days of delivery, and your order was delivered on June 29, 2023—well beyond that window.

If you have any other concerns (e.g., a damaged item) or need further assistanc

[Trace(trace_id=tr-b30aea612b23c809c028f3509902401a), Trace(trace_id=tr-5b6dba5a4e592bfdd099dd14f445da49)]

### Bonus — Out-of-Scope Rejection #2
(Required: at least 2 out-of-scope rejections in the submission)

In [0]:
with mlflow.start_run(run_name="trace_bonus_out_of_scope_2"):
    q6 = "What's the best laptop I should buy for college?"
    print(f"Question: {q6}\n")
    r6 = run_nova(q6, SMALL_MODEL, trace_name="trace_oos_laptop")
    print(f"Nova: {r6}")
    mlflow.log_param("question", q6)
    mlflow.log_param("model", SMALL_MODEL)
    mlflow.log_text(r6, "response.txt")

Question: What's the best laptop I should buy for college?

Nova: I’m sorry, but I can’t help with that. If you have any questions about Nestwell orders or our return and shipping policies, feel free to ask!


Trace(trace_id=tr-57f1770fe6294675844c874f34d79c56)

## 6. Agent Summary

| Trace | Question Type | Model(s) |
|---|---|---|
| 1 | Order status — shipped/in-transit | 20b |
| 2 | Return eligibility — eligible order | 20b |
| 3 | Policy question — shipping costs | 20b |
| 4 | Out-of-scope rejection — poem | 20b |
| 5 | Dual-LLM comparison — ineligible return | 20b vs 120b |
| Bonus | Out-of-scope rejection — laptop advice | 20b |

All traces are logged to the MLflow experiment: `{EXPERIMENT_NAME}`

The evaluation notebook (Notebook 3) will score these traces using
MLflow's built-in LLM judges and our custom alignment judge.